# Phase 3 — a single investigation agent

Does evidence-dependent iterative search add answer quality beyond the frozen Phase 2
retriever R = D3, and at what cost? Everything here was preregistered (`PREREGISTRATION.md`
Phase 3 and amendment A3) before any agent output existed.

**The agent** is a LangGraph `StateGraph` over the frozen D3 backend:

- **Search 1** is the question itself, and returns D3's top 10: exactly R's context.
- **`assess`** (`assess_v2`) decides whether to stop or which query to search next.
- **Follow-up searches** each add the top 5 unseen paragraphs.
- **`answer`** is the frozen `answer_v2`.
- **Budget:** at most 3 searches, 20 paragraphs and 3 LLM calls.

**Data:** `dev` (100 validation questions). All comparisons are paired over the same
questions and exploratory. The blind test splits are not loaded; this notebook only reads
results written by `run_eval.py`.

In [1]:
import collections
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import data
import hops
import manifest
import metrics
import scoring

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6})


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


def fmt(r):
    return f"{r['delta']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]"


dev = data.load_split("dev")
qd = {q.qid: q for q in dev}
ctx = data.load_contexts("dev")
ids = data.doc_id_of()
gold = {q.qid: [ids[g] for g in q.gold] for q in dev}
ag = {r["qid"]: r for r in read_jsonl("results/dev/agent_single.jsonl")}
ag_man = json.load(open("results/dev/agent_single.jsonl.manifest.json"))
print(f"agent records: {len(ag)}; commit {ag_man['comparison_invariants']['code']}; "
      f"new generations recorded as cached: {sum(c['cached'] for r in ag.values() for c in r['calls'])} of "
      f"{sum(len(r['calls']) for r in ag.values())} calls")

agent records: 100; commit {'commit': '37a0466f5032d562b42272eefe2dfb6af521a09c', 'dirty': False}; new generations recorded as cached: 154 of 275 calls


## 1. How the agent behaves (`dev`, n = 100)

Stop reasons, searches, paragraphs, LLM calls and time per question, then the three
behaviour hypotheses:

- **H3-5:** the agent stops at the first `assess` for at least 50% of questions.
- **H3-6:** among questions whose first search already has both gold paragraphs, at most
  40% search again.
- **H3-7:** among questions stopped as `sufficient`, at most 25% are missing a gold
  paragraph.

The `cached` flags only reflect resuming after interruptions: every call is a single
original generation.

In [2]:
rows = []
for q in dev:
    r = ag[q.qid]
    first = set(r["batches"][0])
    rows.append({"qid": q.qid, "type": q.type, "stop": r["stop_reason"], "searches": len(r["batches"]),
                 "calls": len(r["calls"]), "paragraphs": len(r["displayed"]),
                 "wall_s": sum(c["wall_s"] for c in r["calls"]),
                 "prompt_tokens": sum(c["prompt_tokens"] for c in r["calls"]),
                 "output_tokens": sum(c["output_tokens"] for c in r["calls"]),
                 "first_has_both": set(gold[q.qid]) <= first,
                 "final_has_both": set(gold[q.qid]) <= set(r["displayed"]),
                 "first_assess_sufficient": bool(r["assessments"] and r["assessments"][0]["output"]
                                                 and r["assessments"][0]["output"]["sufficient"]),
                 "dropped": sum(len(c["dropped"]) for c in r["calls"])})
beh = pd.DataFrame(rows).set_index("qid")
print("stop reasons:", beh.stop.value_counts().to_dict())
print("searches per question:", beh.searches.value_counts().sort_index().to_dict(),
      f"| LLM calls mean {beh.calls.mean():.2f} | paragraphs in answer context mean {beh.paragraphs.mean():.2f}")
print(f"summed LLM wall per question: p50 {beh.wall_s.median():.1f} s, p95 {beh.wall_s.quantile(0.95):.1f} s; "
      f"prompt tokens mean {beh.prompt_tokens.mean():.0f}; output tokens mean {beh.output_tokens.mean():.0f}; "
      f"overflow drops {beh.dropped.sum()}")

h5 = beh.first_assess_sufficient.mean()
over = beh[beh.first_has_both]
h6 = (over.searches > 1).mean()
suff = beh[beh.stop == "sufficient"]
h7 = (~suff.final_has_both).mean()
print(f"\nH3-5  sufficient at the first assess: {beh.first_assess_sufficient.sum()}/100 = {h5:.2f} "
      f"(predicted >= 0.50) -> {'holds' if h5 >= 0.5 else 'fails'}")
print(f"H3-6  search again although search 1 had both gold: {(over.searches > 1).sum()}/{len(over)} = {h6:.2f} "
      f"(predicted <= 0.40) -> {'holds' if h6 <= 0.4 else 'fails'}")
print(f"H3-7  stopped as sufficient but missing a gold paragraph: {(~suff.final_has_both).sum()}/{len(suff)} = {h7:.2f} "
      f"(predicted <= 0.25) -> {'holds' if h7 <= 0.25 else 'fails'}")

stop reasons: {'max_searches': 44, 'sufficient': 36, 'duplicate_query': 14, 'assess_error': 6}
searches per question: {1: 25, 2: 31, 3: 44} | LLM calls mean 2.75 | paragraphs in answer context mean 15.95
summed LLM wall per question: p50 73.4 s, p95 100.6 s; prompt tokens mean 6236; output tokens mean 254; overflow drops 0

H3-5  sufficient at the first assess: 21/100 = 0.21 (predicted >= 0.50) -> fails
H3-6  search again although search 1 had both gold: 62/84 = 0.74 (predicted <= 0.40) -> fails
H3-7  stopped as sufficient but missing a gold paragraph: 4/36 = 0.11 (predicted <= 0.25) -> holds


The agent is very cautious. It searched again for about three quarters of the questions
whose first search already held both gold paragraphs, so H3-5 and H3-6 fail, as the
`assess` format check had signalled (7 of 20 `sufficient` with full context). When it
does stop as sufficient, it is usually right (H3-7 holds).

## 2. What the extra searches found

- **Useful follow-ups:** follow-up searches that add a gold paragraph missing after
  search 1.
- **Evidence-dependent queries:** following the preregistered operational sign, a
  follow-up query counts as evidence-dependent when it contains a content token (not a
  stop word) that is absent from the question but present in the paragraphs retrieved
  before it.
- **`assess` calibration:** the `sufficient` decision against whether both gold
  paragraphs were present.

In [3]:
from graph import STOPWORDS
from normalize import tok_v1

corpus = data.corpus()
useful = missing_after_1 = n_follow = evidence_dep = dup = 0
examples = []
for q in dev:
    r = ag[q.qid]
    g = set(gold[q.qid])
    missing_after_1 += not g <= set(r["batches"][0])
    qtok = set(tok_v1(q.question))
    for i, (query, batch) in enumerate(zip(r["queries"][1:], r["batches"][1:]), 1):
        n_follow += 1
        seen_before = {d for b in r["batches"][:i] for d in b}
        before_text = set(t for d in seen_before for s in corpus[d].sentences for t in tok_v1(s))
        new_terms = [t for t in tok_v1(query) if t not in qtok and t not in STOPWORDS and t in before_text]
        evidence_dep += bool(new_terms)
        gained = (g - seen_before) & set(batch)
        useful += bool(gained)
        if gained and len(examples) < 3:
            examples.append((q.question.strip(), query, [corpus[d].title for d in gained], new_terms))
    dup += r["stop_reason"] == "duplicate_query"
print(f"follow-up searches: {n_follow}; evidence-dependent queries: {evidence_dep} ({evidence_dep / n_follow:.0%})")
print(f"follow-ups that add a missing gold paragraph: {useful}; questions missing a gold paragraph after search 1: {missing_after_1}")
cum = np.mean([set(gold[q.qid]) <= {d for b in ag[q.qid]['batches'] for d in b} for q in dev])
fin = np.mean([set(gold[q.qid]) <= set(ag[q.qid]["displayed"]) for q in dev])
r1 = np.mean([set(gold[q.qid]) <= set(ag[q.qid]["batches"][0]) for q in dev])
print(f"Joint Recall: search 1 (= R) {r1:.2f}; final context {fin:.2f}; cumulative {cum:.2f}; duplicate-query stops {dup}")
cal = collections.Counter()
for q in dev:
    for i, a in enumerate(ag[q.qid]["assessments"]):
        if a["output"] is None:
            continue
        seen = {d for b in ag[q.qid]["batches"][:i + 1] for d in b}
        cal[(a["output"]["sufficient"], set(gold[q.qid]) <= seen)] += 1
print("assess calibration (sufficient?, both gold present?):", dict(cal))
for ex in examples:
    print(f"\nQ: {ex[0]}\n  follow-up query: {ex[1]!r}  -> added gold {ex[2]}  (new evidence terms {ex[3]})")

follow-up searches: 119; evidence-dependent queries: 103 (87%)
follow-ups that add a missing gold paragraph: 10; questions missing a gold paragraph after search 1: 16
Joint Recall: search 1 (= R) 0.84; final context 0.93; cumulative 0.93; duplicate-query stops 14
assess calibration (sufficient?, both gold present?): {(False, True): 119, (True, True): 32, (False, False): 14, (True, False): 4}

Q: "Coming Over" is the second Japanese single album by a boy group that debuted in what year?
  follow-up query: 'EXO debut year'  -> added gold ['Exo (band)']  (new evidence terms ['exo', 'debut'])

Q: What is the surname of the Italian composer of the works for which the baritone Giovanni Polese was celebrated?
  follow-up query: 'Giuseppe Verdi surname'  -> added gold ['Giuseppe Verdi']  (new evidence terms ['giuseppe', 'verdi'])

Q: William Pery served as Captain of the Yeomen of the Guard under the lord who served in what position from 1865 to 1868?
  follow-up query: 'Henry Cadogan 4th Earl

The iteration is evidence-driven. 87% of the 119 follow-up queries use a term taken from
the retrieved paragraphs rather than the question: for example "EXO debut year", where
EXO is the bridge entity found in the first paragraph.  16 questions lacked at least one gold paragraph after search 1. Follow-ups added a missing paragraph in 10 of them, and 9 finished with both; the tenth had lacked both and recovered only one. Final-context Joint Recall rose from R's 0.84 to 0.93.

The stopping decision is poorly calibrated. 119 times the agent judged the evidence
insufficient while both gold paragraphs were already present, against 32 correct
"sufficient" judgements. The third example also shows the operational sign is crude: its
query named the wrong entity (Henry Cadogan) and still retrieved the gold paragraph.

## 3. Answer quality: agent against R, the matched-context control and BM25

- **The runs:**
  - **R:** the D3 K = 10 run, replayed from the cache under the Phase 3 code.
  - **Matched-context control:** D3 RAG at K = K_match = 16, the rounded mean of the
    agent's answer-context size.
  - **BM25:** the Phase 1 baseline replay.
- **Manifests:** all four pass the final-comparison rule: the same clean commit
  (`37a0466`), one evaluation-code hash, and the answer prompt, schema, validator,
  renderer and system prompt asserted equal.
- **What the matched control means:** it is an approximate, global context-size control,
  not a per-question match.

In [4]:
runs = {"BM25 K=10": "bm25_k10_p3replay", "R = D3 K=10": "d3_k10_p3replay", "D3 K=16 (matched)": "d3_k16", "agent": "agent_single"}
mans = {n: json.load(open(f"results/dev/{f}.jsonl.manifest.json")) for n, f in runs.items()}
SHARED = ["answer_prompt_hash", "schema_hash", "answer_validator_hash", "system_prompt_hash", "renderer_hash"]
for n in list(runs)[:-1]:
    manifest.check_comparable(mans["agent"], mans[n], shared=SHARED, final=True)
recs = {n: {r["qid"]: r for r in read_jsonl(f"results/dev/{f}.jsonl")} for n, f in runs.items()}
sc = {n: {q: scoring.score(qd[q], r) for q, r in recs[n].items()} for n in runs}
MET = {"f1": "answer F1", "em": "EM", "ctx_joint_recall": "context Joint Recall", "sp_strict_f1": "evidence F1 (strict)",
       "joint_official_f1": "joint F1 (official)", "supported": "supported rate", "all_valid_citations": "all citations valid"}
tab = {runs_name: {label: np.mean([s[m] for s in sc[runs_name].values()]) for m, label in MET.items()} for runs_name in runs}
display(pd.DataFrame(tab).round(3))
print(f"displayed paragraphs (mean): " + ", ".join(f"{n} {np.mean([len(r['displayed']) for r in recs[n].values()]):.2f}" for n in runs))
for other in ("R = D3 K=10", "D3 K=16 (matched)", "BM25 K=10"):
    line = []
    for m in ("f1", "ctx_joint_recall", "sp_strict_f1"):
        r = metrics.paired_bootstrap({q: s[m] for q, s in sc["agent"].items()}, {q: s[m] for q, s in sc[other].items()})
        line.append(f"{MET[m]} {fmt(r)}")
    print(f"agent - {other:18s}: " + " | ".join(line))

,BM25 K=10,R = D3 K=10,D3 K=16 (matched),agent
answer F1,0.525,0.629,0.628,0.609
EM,0.410,0.470,0.510,0.470
context Joint Recall,0.620,0.840,0.910,0.930
evidence F1 (strict),0.327,0.453,0.433,0.432
joint F1 (official),0.245,0.335,0.305,0.314
supported rate,0.750,0.870,0.850,0.870
all citations valid,0.820,0.920,0.910,0.880


displayed paragraphs (mean): BM25 K=10 10.00, R = D3 K=10 10.00, D3 K=16 (matched) 16.00, agent 15.95
agent - R = D3 K=10       : answer F1 -0.020 [-0.080, +0.040] | context Joint Recall +0.090 [+0.040, +0.150] | evidence F1 (strict) -0.022 [-0.079, +0.037]
agent - D3 K=16 (matched) : answer F1 -0.019 [-0.081, +0.042] | context Joint Recall +0.020 [-0.030, +0.070] | evidence F1 (strict) -0.001 [-0.063, +0.061]
agent - BM25 K=10         : answer F1 +0.084 [-0.003, +0.175] | context Joint Recall +0.310 [+0.220, +0.400] | evidence F1 (strict) +0.105 [+0.026, +0.184]


## 4. The preregistered Phase 3 hypotheses (exploratory, `dev`, n = 100)

Question-level quality differences (H3-1 to H3-4) are reported with 95% bootstrap
intervals. A threshold prediction holds only when the whole interval clears the
threshold, and an "above zero" prediction is judged on its point estimate. Behavioural
shares (H3-5 to H3-7) and the latency ratio (H3-8) are reported raw, without intervals,
as the preregistration specifies. H3-2 uses an unpaired bootstrap within question types; H3-8 compares the median of each
question's summed agent LLM time with the median of R's stored answer-call time.

In [5]:
def paired(m, a, b, subset=None):
    qs = [q for q in sc[a] if subset is None or qd[q].type == subset]
    return metrics.paired_bootstrap({q: sc[a][q][m] for q in qs}, {q: sc[b][q][m] for q in qs})


h1 = paired("ctx_joint_recall", "agent", "R = D3 K=10")
diff = {t: np.array([sc["agent"][q]["ctx_joint_recall"] - sc["R = D3 K=10"][q]["ctx_joint_recall"]
                     for q in sc["agent"] if qd[q].type == t]) for t in ("bridge", "comparison")}
rng = np.random.default_rng(0)
boot2 = [rng.choice(diff["bridge"], diff["bridge"].size).mean() - rng.choice(diff["comparison"], diff["comparison"].size).mean()
         for _ in range(10_000)]
h2 = diff["bridge"].mean() - diff["comparison"].mean()
h3 = paired("f1", "agent", "R = D3 K=10")
h4 = paired("f1", "agent", "D3 K=16 (matched)")
agent_wall = np.median([sum(c["wall_s"] for c in recs["agent"][q]["calls"]) for q in recs["agent"]])
r_wall = np.median([recs["R = D3 K=10"][q]["llm"]["wall_s"] for q in recs["R = D3 K=10"]])
hyp = pd.DataFrame([
    ("H3-1", "agent - R, context Joint Recall >= +0.03", fmt(h1), "holds" if h1["ci_low"] >= 0.03 else "not shown"),
    ("H3-2", "(delta bridge - delta comparison) > 0", f"{h2:+.3f} [{np.percentile(boot2, 2.5):+.3f}, {np.percentile(boot2, 97.5):+.3f}] "
     f"(bridge {diff['bridge'].mean():+.3f}, n=80; comparison {diff['comparison'].mean():+.3f}, n=20)",
     "holds" if np.percentile(boot2, 2.5) > 0 else "not shown"),
    ("H3-3", "agent F1 - R F1 > 0 (point)", fmt(h3), "holds" if h3["delta"] > 0 else "fails (point estimate below 0)"),
    ("H3-4", "agent F1 - matched-context F1 > 0", fmt(h4), "holds" if h4["delta"] > 0 else "fails (point estimate below 0)"),
    ("H3-5", "sufficient at the first assess >= 50%", f"{beh.first_assess_sufficient.mean():.2f}", "fails"),
    ("H3-6", "searches again although search 1 had both gold <= 40%", f"{h6:.2f} ({(over.searches > 1).sum()}/{len(over)})", "fails"),
    ("H3-7", "stopped as sufficient but missing gold <= 25%", f"{h7:.2f} ({(~suff.final_has_both).sum()}/{len(suff)})", "holds"),
    ("H3-8", "median summed agent LLM time >= 1.8x R's", f"{agent_wall:.1f} s vs {r_wall:.1f} s = {agent_wall / r_wall:.2f}x",
     "holds" if agent_wall / r_wall >= 1.8 else "fails"),
], columns=["id", "prediction", "estimate [95% CI where applicable]", "result"]).set_index("id")
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    display(hyp)
for t in ("bridge", "comparison"):
    print(f"answer F1 {t:10s}: " + " | ".join(f"{n} {np.mean([sc[n][q]['f1'] for q in sc[n] if qd[q].type == t]):.3f}" for n in runs)
          + f" | agent - R {fmt(paired('f1', 'agent', 'R = D3 K=10', t))}")

,prediction,estimate [95% CI where applicable],result
id,,,
H3-1,"agent - R, context Joint Recall >= +0.03","+0.090 [+0.040, +0.150]",holds
H3-2,(delta bridge - delta comparison) > 0,"-0.075 [-0.250, +0.075] (bridge +0.075, n=80; comparison +0.150, n=20)",not shown
H3-3,agent F1 - R F1 > 0 (point),"-0.020 [-0.080, +0.040]",fails (point estimate below 0)
H3-4,agent F1 - matched-context F1 > 0,"-0.019 [-0.081, +0.042]",fails (point estimate below 0)
H3-5,sufficient at the first assess >= 50%,0.21,fails
H3-6,searches again although search 1 had both gold <= 40%,0.74 (62/84),fails
H3-7,stopped as sufficient but missing gold <= 25%,0.11 (4/36),holds
H3-8,median summed agent LLM time >= 1.8x R's,73.4 s vs 23.3 s = 3.15x,holds


answer F1 bridge    : BM25 K=10 0.486 | R = D3 K=10 0.622 | D3 K=16 (matched) 0.615 | agent 0.581 | agent - R -0.040 [-0.107, +0.025]
answer F1 comparison: BM25 K=10 0.680 | R = D3 K=10 0.659 | D3 K=16 (matched) 0.680 | agent 0.720 | agent - R +0.061 [-0.067, +0.190]


## 5. Why more evidence did not mean better answers (exploratory)

Questions are split by what the agent changed relative to R:

- **recovered:** R's context lacked a gold paragraph and the agent's has both;
- **both had both:** the extra paragraphs were redundant;
- **neither:** both contexts lack a gold paragraph.

In each group, the answer-F1 difference shows whether the extra evidence helped where it
was needed, and whether redundant paragraphs hurt where it was not.

In [6]:
groups = collections.defaultdict(list)
for q in sc["agent"]:
    a, r = sc["agent"][q]["ctx_joint_recall"], sc["R = D3 K=10"][q]["ctx_joint_recall"]
    g = "recovered" if a > r else "lost" if a < r else "both had both" if a == 1 else "neither"
    groups[g].append(q)
for g, qs in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    d = np.array([sc["agent"][q]["f1"] - sc["R = D3 K=10"][q]["f1"] for q in qs])
    print(f"{g:14s} n = {len(qs):3d}  agent F1 {np.mean([sc['agent'][q]['f1'] for q in qs]):.3f}  "
          f"R F1 {np.mean([sc['R = D3 K=10'][q]['f1'] for q in qs]):.3f}  difference {d.mean():+.3f}  "
          f"(better {int((d > 0).sum())}, worse {int((d < 0).sum())}, same {int((d == 0).sum())})")
bb = groups["both had both"]
changed = [q for q in bb if recs["agent"][q]["output"]["answer"] != recs["R = D3 K=10"][q]["output"]["answer"]]
print(f"\nin 'both had both', the answer text changed for {len(changed)} of {len(bb)} questions")
for q in [q for q in changed if sc["agent"][q]["f1"] < sc["R = D3 K=10"][q]["f1"]][:3]:
    print(f"  {qd[q].question.strip()[:95]}\n    gold {qd[q].answer!r} | R {recs['R = D3 K=10'][q]['output']['answer']!r} | "
          f"agent {recs['agent'][q]['output']['answer']!r} ({len(recs['agent'][q]['displayed'])} paragraphs)")

both had both  n =  84  agent F1 0.674  R F1 0.707  difference -0.033  (better 4, worse 8, same 72)
recovered      n =   9  agent F1 0.370  R F1 0.259  difference +0.111  (better 2, worse 1, same 6)
neither        n =   7  agent F1 0.143  R F1 0.171  difference -0.029  (better 0, worse 1, same 6)

in 'both had both', the answer text changed for 15 of 84 questions
  Besides the founder of Led Zeppelin, and Bert Jansch, what other to British musician was heavil
    gold 'Eric Clapton' | R 'Eric Clapton' | agent 'Bert Jansch' (15 paragraphs)
  What sort of activist is the author of The Irrational Atheist?
    gold 'alt-right' | R 'alt-right activist' | agent 'writer' (20 paragraphs)
  The Ameican Jazz saxophonist nicked named the "Hawk", once played with what musician who died i
    gold 'Shep Meyers' | R 'Shep Meyers' | agent 'Coleman Hawkins' (20 paragraphs)


Where the agent recovered missing evidence (9 questions), answers improved (+0.111 F1).
Where R already had both gold paragraphs (84 questions), F1 was 0.033 lower after
redundant paragraphs were added (8 worse, 4 better), consistent with reader distraction:
for example "Bert Jansch" instead of "Eric Clapton", or "Coleman Hawkins" (the subject)
instead of "Shep Meyers". This subgroup analysis is exploratory and does not establish
causation. Because R already had full evidence for 84% of
questions, the redundant-context cost outweighs the recovery gain.

## 6. Phase 3 summary

| `dev`, n = 100 | BM25 K = 10 | R = D3 K = 10 | D3 K = 16 (matched) | Agent |
|---|---|---|---|---|
| Answer F1 | 0.525 | **0.629** | 0.628 | 0.609 |
| Context Joint Recall | 0.62 | 0.84 | 0.91 | **0.93** |
| LLM calls per question | 1 | 1 | 1 | 2.75 |
| Median LLM time per question | about 21 s | 23 s | about 35 s | 73 s |

**Does iterative search add answer quality beyond R? Not in this setting.**

- **Evidence goes up, answers don't.** The agent raises context Joint Recall (+0.090
  [+0.040, +0.150], H3-1) with genuinely evidence-dependent queries. Its answer F1 is
  −0.020 [−0.080, +0.040] against R, and −0.019 [−0.081, +0.042] against simply giving
  RAG 16 paragraphs.
- **Recovery does help,** where R lacked the evidence (9 questions, +0.11).
- **F1 is lower after redundant paragraphs** where R already had both gold paragraphs
  (84 questions, −0.03; consistent with distraction, not proof of it).
- **The stopping decision is the weak component.** The agent searched again for 74% of
  the questions whose first search already contained both gold paragraphs (H3-6 fails), and stopped at
  the first `assess` for only 21% (H3-5 fails).
- **It costs 3.15× R's LLM time** (H3-8).
- **Hypotheses:** H3-1, H3-7 and H3-8 hold; H3-3, H3-4, H3-5 and H3-6 fail; H3-2 is not
  shown (its point estimate runs the other way).

A strong retriever leaves little for iteration to find: R already had both gold
paragraphs for 84% of these questions. The agent's value would lie in **routing**:
searching again only when the first context is insufficient. That requires a better
sufficiency judgement than this 7B model's `assess` step provides. The result applies to
this model, this prompt and this budget.